In [ ]:
##==============================================================================
## Armado programático de TP | @file /tp_final.ipynb
##==============================================================================
## No se requiere importar aqui fgrLib porque se la importa desde el compile.sh
##==============================================================================


import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from IPython.display import display
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
from pathlib import Path

import sys
import pyautogui

import importlib

import re	## Usada para parsear los archivos de mediciones.

from pylab import rcParams

In [ ]:
img_dir = "./img/"
os.makedirs(img_dir, exist_ok = True)

In [ ]:
#===============================================================================
# Algunas personalizaciones globales para los graficos
#===============================================================================

plt.rcdefaults() # Resetear a los valores por defecto.

col_def = plt.rcParams["axes.prop_cycle"].by_key()["color"]		# 10 col originales.

col_def[1] = "#ff5500"

plt.rcParams["axes.prop_cycle"] = plt.cycler(color = col_def)	# Aplicar lista.

rcParams["figure.figsize"] = 8, 5		# Tan solo cambio para ajuste de cantidad de hojas ocupadas.
rcParams["savefig.format"] = "png"
rcParams["savefig.dpi"] = 300

plt.rcParams["axes.grid"] = True		# Activa la grilla automáticamente en todos los plots
plt.rcParams["grid.linestyle"] = "-"	# Fuerza a que sea línea sólida
plt.rcParams["grid.alpha"] = 0.5		# Ajusta la opacidad opcionalmente para que no sea tan invasiva

alpha_plot_global = 0.15

In [ ]:
#===============================================================================
# Variables que arman la carátula
#===============================================================================

asignatura =  "Diseño de Circuitos Integrados Digitales 1"
tp_nro = 1
tp_tit = "Lint"
bimestre = 4
year = 2026
fecha_entrega = "11-09-2026"

In [ ]:
# ==============================================================================
# 0000_0000 | Carátula
# ==============================================================================

enu_0000_0000 = f"""<!-- fgrLib.export_this_output("0000_0000.tex") -->

#### <center>Facultad de Ingeniería de la Universidad de Buenos Aires</center>
# <center>Carrera de Especialización en Microelectrónica</center>

## <b>{asignatura}</b>
# Trabajo Práctico N°{tp_nro}: {tp_tit}
#### <b>Estudiante:</b> Federico Ramos (<font name="tt"><color name="blue">federico.g.ramos@gmail.com</color></font>)
#### <b>Año de cursada:</b> {bimestre}B-{year}
#### <b>Fecha de entrega:</b> {fecha_entrega}
</br>
"""

print(enu_0000_0000)


In [ ]:
# ==============================================================================
# 0010_enunciado_0000 | Enunciado
# ==============================================================================

enu_0010_enunciado_0000 = f"""<!-- fgrLib.export_this_output("0010_enunciado_0000.tex") -->

Aprender a usar herramientas de LINT para encontrar problemas en el código RTL. Asegurar un RTL simulable e implementable.

Ejecutar HAL sobre un diseño RTL que implementa la funcionalidad de un slave de protocolo I2C.

Analizar warnings y reescribir código respetando funcionalidad original.

Entrega:
* RTL modificado sin warnings críticos.
* Breve informe de problemas encontrados (por ej: latch inferido en linea 124, race condition en línea 63).

"""

print(enu_0010_enunciado_0000)

In [ ]:
# ==============================================================================
# 0010_answer_0000 | serialInterface.v
# ==============================================================================

ans_0010_answer_0000 = f"""<!-- fgrLib.export_this_output("0010_answer_0000.tex") -->

<problem>

* Luego de hacer las modificaciones en los archivos, le hago un diff con el original, para marcar las diferencias, o lineas agregadas y las eliminadas.

#### **Explicaciones de las ediciones de codigo realizadas**

* Utilizo el mismo archivo fuente para indicar el motivo de cada cambio utilizando comentarios.

#### **Colores de referencia**

* <font name="tt"><color name="gray">gris claro</color></font>: las lineas sin cambio para dar algo de contexto al codigo que se muestra.

* <font name="tt"><color name="diffDelText">rojo</color></font>: lineas eliminadas.

* <font name="tt"><color name="diffAddText">verde</color></font>: lineas agregadas, las cuales pueden incluir la descripcion del motivo del cambio (en un comentario).

* <font name="tt"><color name="pyKeyword">azul</color></font>: indica el numero de linea que ese está alli mostrando.

# **Archivo: <font name="tt">serialInterface.v</font>**

<code name="diff">
$ git diff --no-index -U1 --color=always  ./fuentes_originales/rtl/serialInterface.v ./fuentes/rtl/serialInterface.v
diff --git a/./fuentes_originales/rtl/serialInterface.v b/./fuentes/rtl/serialInterface.v
index aa65380..bccdaee 100644
--- a/./fuentes_originales/rtl/serialInterface.v
+++ b/./fuentes/rtl/serialInterface.v
@@ -105,3 +105,3 @@ localparam WRITE_CLR_WR = 5'b1110;
 localparam WRITE_CLR_ST_STOP = 5'b1111;
-localparam END = 5'b11111;
+// Si no se usa puede borrarse.
 
@@ -122,2 +122,3 @@ always @(*) begin
   next_regAddr = regAddr;
+  next_writeEn = writeEn; // tp1: agregado para evitar latches.
 
@@ -126,3 +127,3 @@ always @(*) begin
       next_streamSt = `STREAM_IDLE;
-      next_txData = 12'h00;
+      next_txData = 8'h00; // Aqui sobran bits.
       next_rxData = 8'h00;
@@ -140,3 +141,3 @@ always @(*) begin
         next_txData = dataIn;
-        next_regAddr = regAddr + 8'b1;
+        next_regAddr = (regAddr + 8'b1) & 8'hFF; // Trunco a 8 bits.
         next_bitCnt = 3'b001;
@@ -189,3 +190,3 @@ always @(*) begin
         NextState_SISt = READ_RD_LOOP;
-        next_bitCnt = bitCnt + 3'b1;
+        next_bitCnt = (bitCnt + 3'b1) & 3'b111; // Trunco a 3 bits.
       end
@@ -242,3 +243,3 @@ always @(*) begin
         if (startStopDetState == `NULL_DET) begin
-          next_bitCnt = bitCnt + 3'b1;
+          next_bitCnt = (bitCnt + 3'b1) & 3'b111; // Trunco a 3 bits.
         end else if (startStopDetState == `START_DET) begin
@@ -256,3 +257,3 @@ always @(*) begin
         if (startStopDetState == `NULL_DET) begin
-          next_bitCnt = bitCnt + 3'b1;
+          next_bitCnt = (bitCnt + 3'b1) & 3'b111; // Trunco a 3 bits.
         end else if (startStopDetState == `START_DET) begin
@@ -328,3 +329,3 @@ always @(*) begin
             NextState_SISt = WRITE_ST_LOOP;
-            next_bitCnt = bitCnt + 3'b1;
+            next_bitCnt = (bitCnt + 3'b1) & 3'b111; // Trunco a 3 bits.
             next_streamSt = streamSt;
@@ -403,3 +404,3 @@ always @(*) begin
       if (writeEn == 1'b1) begin
-        next_regAddr = regAddr + 8'b1;
+        next_regAddr = (regAddr + 8'b1) & 8'hFF; // Trunco a 8 bits.
       end else begin
@@ -429,14 +430,3 @@ always @(*) begin
     end
-    END: begin
-      next_streamSt = `STREAM_IDLE;
-      next_txData = 12'h00;
-      next_rxData = 8'h00;
-      next_sdaOut = 1'b1;
-      next_writeEn = 1'b0;
-      next_dataOut = 8'h00;
-      next_bitCnt = 3'b000;
-      next_clearStartStopDet = 1'b0;
-      next_regAddr = regAddr;
-      NextState_SISt = START;
-    end
+    // Si no se usa puede borrarse.
     default: begin
</code>

</problem>

"""

print(ans_0010_answer_0000)

In [ ]:
# ==============================================================================
# 0010_answer_0001 | registerInterface.v
# ==============================================================================

ans_0010_answer_0001 = f"""<!-- fgrLib.export_this_output("0010_answer_0001.tex") -->

# **Archivo: <font name="tt">registerInterface.v</font>**

<problem>

<code name="diff">
$ git diff --no-index -U2 --color=always  ./fuentes_originales/rtl/registerInterface.v ./fuentes/rtl/registerInterface.v
diff --git a/./fuentes_originales/rtl/registerInterface.v b/./fuentes/rtl/registerInterface.v
index 64edcb6..def1a4b 100644
--- a/./fuentes_originales/rtl/registerInterface.v
+++ b/./fuentes/rtl/registerInterface.v
@@ -83,5 +83,5 @@ reg [7:0] myReg1;
 reg [7:0] myReg2;
 reg [7:0] myReg3;
-wire write_done;
+// wire write_done; ==> No usado, se puede borrar.
 
 // --- I2C Read
@@ -96,5 +96,5 @@ always @(posedge clk) begin
     8'h06: dataOut <= myReg6;  
     8'h07: dataOut <= myReg7;  
-    8'hxx: dataOut <= 8'hxx;  
+    // 8'hxx: dataOut <= 8'hxx; <== esto hay que eliminarlo. No es sintetizable.
     default: dataOut <= 8'h00;
   endcase
</code>

</problem>

"""

print(ans_0010_answer_0001)

In [ ]:
# ==============================================================================
# 0010_answer_0002 | i2cSlave.v
# ==============================================================================

ans_0010_answer_0002 = f"""<!-- fgrLib.export_this_output("0010_answer_0002.tex") -->

<problem>

# **Archivo: <font name="tt">i2cSlave.v</font>**


<code name="diff">
$ git diff --no-index -U2 --color=always  ./fuentes_originales/rtl/i2cSlave.v ./fuentes/rtl/i2cSlave.v
diff --git a/./fuentes_originales/rtl/i2cSlave.v b/./fuentes/rtl/i2cSlave.v
index e7b16a2..138fb95 100644
--- a/./fuentes_originales/rtl/i2cSlave.v
+++ b/./fuentes/rtl/i2cSlave.v
@@ -63,5 +63,5 @@ module i2cSlave (
 input clk;
 input rst;
-inout sda;
+inout sda;  // En clase se indico que este lo dejemos asi, tira el warning de que es bidireccional y puede traer problemas. Se explico que lo ideal seria ponerlos separados.
 input scl;
 output [7:0] myReg0;
@@ -94,4 +94,5 @@ reg [1:0] rstPipe;
 reg startEdgeDet;
 reg rst_serial_interface;
+wire rstSyncToClk; // Faltaba esta declaracion.
   
-always @(rstSyncToClk) begin
-  rst_serial_interface = rstSyncToClk | startEdgeDet;
+always @(posedge clk) begin // Se pasa a secuencial.
+  rst_serial_interface <= rstSyncToClk | startEdgeDet;
 end
</code>

# **Resumen de de lo que queda**


<latex_fig src="/0010/fin.png" cap="Captura de pantalla de mensajes restantes (indica warnings = 4)." lbl="fig:fin"/>


<latex_table cap="Resumen jerárquico de componentes, conteo de transistores y áreas de canal estimadas." lbl="tab:jerarquiaComponentes">
\\begin{{center}}
\\begin{{tabularx}}{{\\textwidth}}{{|CX|CX|CX|CX|CX|}}
\\hline
\\rowcolor{{tabColorHdr_085}}
    \\textnormal{{Bloque / Celda}} & 
    \\textnormal{{Componentes Internos Directos}} & 
    \\textnormal{{Transistores por Instancia}} & 
    \\textnormal{{Área de Canal por Instancia ($\\mu m^2$)}} & 
    \\textnormal{{Descripción jerárquica dentro del sistema}} \\\\ \\hline
\\rowcolor{{tabColor}}tgate & 1 PMOS + 1 NMOS & 2 & 0.0108 & Compuerta de transmisión. \\\\ \\hline
\\rowcolor{{tabColor}}inv & 1 PMOS + 1 NMOS & 2 & 0.0108 & Inversor lógico básico. \\\\ \\hline
\\rowcolor{{tabColor}}c2mos\\_latch & 2 PMOS + 2 NMOS & 4 & 0.0216 & Latch dinámico $C^2MOS$. \\\\ \\hline
\\rowcolor{{tabColor}}dinamic\\_flop & 2 tgate + 3 inv & 10 & 0.0540 & Flip-flop dinámico. \\\\ \\hline
\\rowcolor{{tabColor}}dinamic\\_flop\\_c2mos & 2 c2mos\\_latch + 1 inv & 10 & 0.0540 & Flip-flop síncrono dinámico ($2\\times4 + 1\\times2$). \\\\ \\hline
\\rowcolor{{tabColor}}static\\_latch & 4 inv + 2 tgate & 12 & 0.0648 & Latch estático. \\\\ \\hline
\\rowcolor{{tabColor}}static\\_flop & 4 tgate + 7 inv & 22 & 0.1188 & Flip-flop estático. \\\\ \\hline
\\rowcolor{{tabColor}}mux\\_2to1\\_v2 & 3 inv + 2 tgate & 10 & 0.0540 & Multiplexor 2 a 1. \\\\ \\hline
\\rowcolor{{tabColor}}ser\\_2to1\\_V2 & 1 inv + 2 dinamic\\_flop\\_c2mos + 1 static\\_latch + 1 mux\\_2to1\\_v2 & 44 & 0.2376 & Unidad de serialización de 2 a 1. \\\\ \\hline
\\rowcolor{{tabColor}}core\\_schematic & 15 ser\\_2to1\\_V2 + 1 dinamic\\_flop + 15 inv & 700 & 3.7800 & Árbol serializador global de 16 bits. \\\\ \\hline
\\rowcolor{{tabColor}}clk\\_div & 9 static\\_flop + 301 inv + 5 resistor & 800 & 4.3200 & Árbol de distribución y división de reloj. \\\\ \\hline
\\rowcolor{{tabColorHdr_085}}serializer (TOP LEVEL) & 1 core\\_schematic + 1 clk\\_div & 1500 & 8.1000 & Circuito Integrado Serializador Completo de 16-bits. \\\\ \\hline
\\end{{tabularx}}
\\end{{center}}
</latex_table>


</problem>
"""

print(ans_0010_answer_0002)